# Head rows builder (Kaggle CPU, internet ON)
Exploratory (no gate). Downloads everything inside Kaggle (HuggingFace hub). Nothing is read from or uploaded from the local machine.
Sources: nebius/SWE-rebench-openhands-trajectories, SWE-bench/SWE-smith-trajectories (tool-* shards only), DSFFGFG456/fable-5-coding-and-debugging-traces (pinned).
Each decision row = gold action + 2-3 synthetic hard negatives (executed=false, no outcome) + NONE. Labels: `resolved` (real outcome) only for nebius and swesmith; fable5 has none.
Contamination: repos seen by Coding_Decision_Agent (S4MPL3BI4S/coding-decision-cases) are tagged and never placed in TEST (they may be in TRAIN/CAL). Shortcut check runs at the end.
Run with the Kaggle "Save version" so `/kaggle/working/head_rows/` becomes an output the sampler notebook can attach.

In [ ]:
import os, sys, json, re, glob, hashlib, random, subprocess, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pyarrow", "scikit-learn"], check=True)
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download, HfApi
api = HfApi()
OUT = "/kaggle/working/head_rows"; os.makedirs(OUT, exist_ok=True)
SEED = 20261004
CFG = dict(  # trajectory caps per source: unmeasured throughput guess, edit before running; None = all
    cap_traj={"nebius": 4000, "swesmith": 3000}, k_mid=3, state_chars=5000, act_chars=600, task_chars=1500, turn_chars=1200,
    test_mod=20, cal_lo=20, cal_hi=30, shortcut_margin=0.15)
SRC = {
    "nebius":   dict(repo="nebius/SWE-rebench-openhands-trajectories", rev=None, pat=["*.parquet"]),
    "swesmith": dict(repo="SWE-bench/SWE-smith-trajectories", rev=None, pat=["data/tool-*.parquet"]),
    "fable5":   dict(repo="DSFFGFG456/fable-5-coding-and-debugging-traces", rev="dd283de3f8b7ab5ecb2f4a26a22717f162d71557", pat=["data/train-*.parquet"]),
    "cases":    dict(repo="S4MPL3BI4S/coding-decision-cases", rev="2ebe9040038897d8e3037c672d0dccb958a04a1c", pat=["data/*.parquet"]),
}
import fnmatch
from huggingface_hub import hf_hub_download
NEED = {"nebius": {"trajectory_id", "trajectory", "resolved", "repo", "instance_id"}, "swesmith": {"messages", "resolved", "instance_id", "traj_id"},
        "fable5": {"messages", "target_message_index", "assistant_step", "source_trajectory_id"}, "cases": set()}
for k, s in SRC.items():     # fail fast: one shard per source, columns asserted before any large download
    s["rev"] = api.dataset_info(s["repo"], revision=s["rev"]).sha
    names = [f for f in api.list_repo_files(s["repo"], repo_type="dataset", revision=s["rev"]) if any(fnmatch.fnmatch(f, p) for p in s["pat"])]
    assert names, (k, "no files match", s["pat"])
    one = pq.ParquetFile(hf_hub_download(s["repo"], names[0], repo_type="dataset", revision=s["rev"]))
    cols = one.schema_arrow.names
    print(k, s["repo"], s["rev"][:10], len(names), "files | columns:", cols)
    assert NEED[k] <= set(cols), (k, "missing columns", NEED[k] - set(cols))
    if k == "fable5":
        r = one.read_row_group(0).slice(0, 1).to_pylist()[0]
        m = r["messages"] if isinstance(r["messages"], list) else json.loads(r["messages"])
        assert m[r["target_message_index"]]["role"] == "assistant", "fable target message is not an assistant turn"
print("schema ok")

In [ ]:
PATH = {}
for k, s in SRC.items():
    PATH[k] = snapshot_download(s["repo"], repo_type="dataset", revision=s["rev"], allow_patterns=s["pat"])
    fl = sorted(glob.glob(PATH[k] + "/**/*.parquet", recursive=True))
    print(k, len(fl), "files", round(sum(os.path.getsize(f) for f in fl) / 2**30, 2), "GB")
    s["files"] = [os.path.relpath(f, PATH[k]) for f in fl]
def files(k): return sorted(glob.glob(PATH[k] + "/**/*.parquet", recursive=True))
cases_cols = {c for f in files("cases") for c in pq.ParquetFile(f).schema_arrow.names}
print("cases", sorted(cases_cols))

## Contaminated repos (derived here from coding-decision-cases; never exported as a local file)

In [ ]:
def norm_repo(v):
    v = str(v)
    if "__" in v: v = v.split(".")[0].replace("__", "/", 1)   # instance id owner__repo.hash... -> owner/repo
    return v.strip().lower()
assert {"source", "group"} <= cases_cols, cases_cols
cases_repos = set()      # rows built from nebius/SWE-agent-trajectories carry the repo as group = owner__repo
for f in files("cases"):
    for b in pq.ParquetFile(f).iter_batches(batch_size=4096, columns=["source", "group"]):
        for src, grp in zip(b.column(0).to_pylist(), b.column(1).to_pylist()):
            if src == "nebius/SWE-agent-trajectories" and grp: cases_repos.add(str(grp).replace("__", "/", 1).strip().lower())
print("cases repos from nebius/SWE-agent-trajectories:", len(cases_repos), "(earlier local scan: 858)")
assert len(cases_repos) > 100

## Normalise trajectories, pick decision steps, build hard negatives

In [ ]:
def jl(x): return json.loads(x) if isinstance(x, str) else x
def jsafe(x):     # tool-call arguments: keep the raw string if it is not valid JSON
    try: return json.loads(x)
    except json.JSONDecodeError: return x
SKIPPED = {"nebius": 0, "swesmith": 0, "fable5": 0}   # records whose JSON could not be parsed; skipped, never repaired
def text_of(m):
    c = m.get("content")
    if isinstance(c, list): c = " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)
    return c or ""
def action_text(m):
    if isinstance(m.get("action"), str) and m["action"].strip(): return m["action"][:CFG["act_chars"]]
    tc = m.get("tool_calls")
    if tc:
        out = []
        for c in tc:
            f = c.get("function", c); a = f.get("arguments")
            out.append({"name": f.get("name"), "arguments": jsafe(a) if isinstance(a, str) and a[:1] in "{[" else a})
        return json.dumps(out[0] if len(out) == 1 else out, ensure_ascii=False)[:CFG["act_chars"]]
    return text_of(m)[:CFG["act_chars"]]
def adapt_msgs(msgs, source):
    msgs = [m for m in jl(msgs) if m["role"] != "system"]
    if source == "swesmith" and not any(m["role"] == "tool" for m in msgs):   # observations come back as user turns after the task
        seen = False; out = []
        for m in msgs:
            r = m["role"]
            if r == "user": r, seen = ("user" if not seen else "tool"), True
            out.append({**m, "role": r})
        return out
    return msgs
FINISH = json.dumps({"name": "finish", "arguments": {"message": "The task is complete."}})
PATHRE = re.compile(r"[\w./-]+\.(?:py|js|ts|md|txt|toml|cfg|json|ya?ml|rs|go|java|c|h|cpp)\b")
NAMERE = re.compile(r'"name": "([^"]+)"')
def negatives(gold, others, rng, n=3):
    got, seen = [], {gold}
    def add(kind, t):
        if t and t.strip() and t not in seen: seen.add(t); got.append((kind, t))
    pool = [o for o in others if o.strip() and o != gold]
    if pool: add("other_step", rng.choice(pool))
    ps = PATHRE.findall(gold); alt = sorted({p for o in others for p in PATHRE.findall(o)} - set(ps))
    if ps and alt: add("wrong_path", gold.replace(ps[0], rng.choice(alt), 1))
    names = sorted({n for o in others for n in NAMERE.findall(o)})
    g = NAMERE.search(gold)
    if g and len(names) > 1: add("wrong_tool", gold.replace(g.group(0), '"name": "%s"' % rng.choice([x for x in names if x != g.group(1)] or names), 1))
    if "finish" not in gold.lower(): add("premature_finish", FINISH)
    rng.shuffle(got)
    return got[:n]
def mk_row(source, tid, repo, resolved, state_text, gold, others, step, n_steps, is_last, rng):
    neg = negatives(gold, others, rng)
    if len(neg) < 2: return None
    cands = [{"id": "gold", "kind": "gold", "text": gold, "executed": True}]
    cands += [{"id": "neg%d" % i, "kind": "hard_neg:" + k, "text": t, "executed": False} for i, (k, t) in enumerate(neg)]
    cands.append({"id": "NONE", "kind": "NONE", "text": "NONE: not enough evidence to choose; gather more information or ask the user.", "executed": False})
    return {"decision_id": "%s:%s:%d" % (source, tid, step), "source": source, "trajectory_id": tid, "repo": repo, "step": step, "n_steps": n_steps, "is_last": is_last,
            "resolved": None if resolved is None else int(resolved), "done": None if resolved is None else int(is_last and resolved), "state_text": state_text,
            "state_hash": hashlib.sha256(state_text.encode()).hexdigest()[:16], "candidates": cands}
def render_state(msgs, upto):
    task = next((text_of(m) for m in msgs if m["role"] == "user"), "")[:CFG["task_chars"]]
    turns = ["%s: %s" % (m["role"], text_of(m)[-CFG["turn_chars"]:]) for m in msgs[max(0, upto - 3):upto]]
    return ("TASK: " + task + "\n\nRECENT:\n" + "\n".join(turns))[-CFG["state_chars"]:]
def traj_rows(source, rec, tid, repo, resolved):
    msgs = adapt_msgs(rec, source)
    steps = [i for i, m in enumerate(msgs) if m["role"] == "assistant"]
    if len(steps) < 3: return []
    rng = random.Random(tid); last = len(steps) - 1
    pick = {last, last - 1} | set(rng.sample(range(0, last - 1), min(CFG["k_mid"], last - 1)))
    acts = [action_text(msgs[i]) for i in steps]
    rows = []
    for s in sorted(pick):
        r = mk_row(source, tid, repo, resolved, render_state(msgs, steps[s]), acts[s], acts[:s] + acts[s + 1:], s, len(steps), s == last, rng)
        if r: rows.append(r)
    return rows
def selftest():
    t = [{"role": "user", "content": "fix bug in a/b.py"}]
    for i, (n, p) in enumerate([("view", "a/b.py"), ("edit", "a/b.py"), ("run", "tests/t.py"), ("finish", "x")]):
        t += [{"role": "assistant", "content": "", "tool_calls": [{"function": {"name": n, "arguments": {"path": p}}}]}, {"role": "tool", "content": "ok"}]
    rows = traj_rows("nebius", t, "t1", "o/r", 1)
    assert rows and all(len(r["candidates"]) >= 4 and r["candidates"][0]["kind"] == "gold" and r["candidates"][-1]["id"] == "NONE" for r in rows)
    assert all(c["text"] != r["candidates"][0]["text"] for r in rows for c in r["candidates"][1:]), "negative equals gold"
selftest(); print("selftest ok")

In [ ]:
def split_of(repo_key, contaminated):
    h = int(hashlib.sha256(repo_key.encode()).hexdigest(), 16) % 100
    if h < CFG["test_mod"] and not contaminated: return "test"
    return "cal" if CFG["cal_lo"] <= h < CFG["cal_hi"] else "train"
rows, stats = [], {}
for source in ("nebius", "swesmith"):
    cap, seen, n_row = CFG["cap_traj"][source], set(), 0
    for f in files(source):
        for b in pq.ParquetFile(f).iter_batches(batch_size=32):
            for rec in b.to_pylist():
                if cap and len(seen) >= cap: break
                iid = rec["instance_id"]
                repo = rec["repo"] if source == "nebius" and rec.get("repo") else norm_repo(iid)
                tid = str(rec.get("trajectory_id") or rec.get("traj_id") or rec.get("traj_hash") or "%s:%s:%d" % (source, iid, len(seen)))
                if tid in seen: continue
                seen.add(tid)
                try:
                    rows += traj_rows(source, rec["trajectory"] if source == "nebius" else rec["messages"], tid, norm_repo(repo), bool(rec["resolved"]))
                except json.JSONDecodeError:
                    SKIPPED[source] += 1
            if cap and len(seen) >= cap: break
        if cap and len(seen) >= cap: break
    stats[source] = dict(trajectories=len(seen), rows=sum(1 for r in rows if r["source"] == source))
# fable5: one row per decision already; negatives come from other decisions of the same trajectory
by_traj = {}
for f in files("fable5"):
    for b in pq.ParquetFile(f).iter_batches(batch_size=64, columns=["messages", "target_message_index", "source_trajectory_id"]):
        for r in b.to_pylist():
            try: msgs = jl(r["messages"])
            except json.JSONDecodeError:
                SKIPPED["fable5"] += 1; continue
            t = int(r["target_message_index"])
            if msgs[t]["role"] == "assistant": by_traj.setdefault(str(r["source_trajectory_id"]), []).append((msgs[:t], action_text(msgs[t])))
n0 = len(rows)
for tid, rs in by_traj.items():
    acts = [a for _, a in rs]
    for j, (prefix, gold) in enumerate(rs):
        pre = [m for m in prefix if m["role"] != "system"]
        row = mk_row("fable5", tid, "fable5/" + tid, None, render_state(pre, len(pre)), gold, acts[:j] + acts[j + 1:], j, len(rs), j == len(rs) - 1, random.Random("%s:%d" % (tid, j)))
        if row: rows.append(row)
stats["fable5"] = dict(trajectories=len(by_traj), rows=len(rows) - n0)
# dedup by state hash, tag contamination + split
seen_h, final = set(), []
for r in rows:
    if r["state_hash"] in seen_h: continue
    seen_h.add(r["state_hash"])
    r["contaminated_repo"] = r["repo"] in cases_repos
    r["split"] = split_of(r["repo"], r["contaminated_repo"])
    final.append(r)
rows = final
print(stats, "| skipped bad-JSON records:", SKIPPED, "| after dedup:", len(rows))
from collections import Counter
print(Counter((r["source"], r["split"]) for r in rows))
print("test rows in contaminated repos:", sum(1 for r in rows if r["split"] == "test" and r["contaminated_repo"]), "(must be 0)")
assert not any(r["split"] == "test" and r["contaminated_repo"] for r in rows)

## Shortcut check: can candidate TEXT alone find the gold? (margin 0.15 written before any result)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
def flat(rs): return [(i, c) for i, r in enumerate(rs) for c in r["candidates"] if c["id"] != "NONE"]
tr = [r for r in rows if r["split"] == "train"]; ev = [r for r in rows if r["split"] == "cal"]
ftr, fev = flat(tr), flat(ev)
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000, sublinear_tf=True)
Xtr = vec.fit_transform([c["text"] for _, c in ftr]); Xev = vec.transform([c["text"] for _, c in fev])
clf = LogisticRegression(max_iter=300).fit(Xtr, [int(c["kind"] == "gold") for _, c in ftr])
sc = clf.decision_function(Xev); best = {}
for (i, c), s in zip(fev, sc):
    if i not in best or s > best[i][0]: best[i] = (s, c["kind"])
acc = float(np.mean([k == "gold" for _, k in best.values()]))
chance = float(np.mean([1.0 / (len(r["candidates"]) - 1) for r in ev]))
per_kind = {}
for r in ev:
    for c in r["candidates"]:
        if c["kind"].startswith("hard_neg"): per_kind[c["kind"]] = per_kind.get(c["kind"], 0) + 1
shortcut = dict(label="exploratory (no gate)", text_only_gold_top1=acc, chance=chance, margin=CFG["shortcut_margin"], flagged=acc > chance + CFG["shortcut_margin"],
                eval_decisions=len(ev), train_decisions=len(tr), neg_kind_counts_in_eval=per_kind)
print(shortcut)

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
with open(OUT + "/rows.jsonl", "x", encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
json.dump(dict(label="exploratory (no gate)", stamp=stamp, seed=SEED, cfg=CFG, sources={k: dict(repo=s["repo"], revision=s["rev"], files=len(s["files"])) for k, s in SRC.items()},
               stats=stats, skipped_bad_json=SKIPPED, rows=len(rows), split_counts={"%s/%s" % k: v for k, v in Counter((r["source"], r["split"]) for r in rows).items()},
               cases_repo_count=len(cases_repos), contaminated_rows=sum(r["contaminated_repo"] for r in rows)), open(OUT + "/manifest.json", "x"), indent=1)
json.dump(shortcut, open(OUT + "/shortcut-check.json", "x"), indent=1)
print("wrote", OUT, "| flagged:", shortcut["flagged"])